<a href="https://colab.research.google.com/github/janeliuPO/MyJuly26_BCCE/blob/main/Understanding_Kd_Biochemistry_Lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Understanding the Dissociation Constant (Kd)
## A Guided Introduction for Biochemistry Students

Welcome! In this notebook, you'll learn about one of the most important concepts in biochemistry: the **dissociation constant (Kd)**.

### What you'll learn:
1. **Part 1:** What Kd is and what it tells you about a biological system
2. **Part 2:** How to interpret binding curves and extract Kd from data
3. **Part 3:** The relationship between Kd and Ka (association constant)
4. **Part 4:** How to fit your own data and create binding curves

### How to use this notebook:
- **Read** each text block carefully — they explain the concepts.
- **Run** each code block by clicking the ▶️ play button on the left, or by pressing `Shift + Enter`.
- **Don't worry** if you've never coded before! Everything is set up for you. You just need to run the cells and change a few numbers here and there.
- **Experiment!** The best way to learn is to change values and see what happens.

---

## 🔧 Setup: Run This First

This cell loads the tools (called *libraries*) we need for plotting and math. **Just run it — you don't need to understand it yet.**

In [ ]:
# These are 'libraries' — pre-made toolkits for doing math and making plots
import numpy as np                  # for math and arrays of numbers
import matplotlib.pyplot as plt     # for making plots
from scipy.optimize import curve_fit  # for fitting curves to data

print("✅ Setup complete! You're ready to go.")

---
# Part 1: What is Kd? Building the Concept

## The Binding Equilibrium

Imagine a **protein (P)** that binds a **ligand (L)** — for example, an enzyme binding its substrate, a receptor binding a hormone, or an antibody binding an antigen. This binding is reversible:

$$ P + L \rightleftharpoons PL $$

At equilibrium, the **dissociation constant Kd** is defined as:

$$ K_d = \frac{[P][L]}{[PL]} $$

where:
- **[P]** = concentration of free (unbound) protein
- **[L]** = concentration of free (unbound) ligand
- **[PL]** = concentration of the protein-ligand complex

## 🧠 What does Kd actually TELL you?

**Kd has units of concentration** (usually molar, M). This is the key to understanding it:

> **Kd is the ligand concentration at which HALF of the protein is bound to ligand.**

Think about this intuitively:
- **Small Kd (e.g., 1 nM)** → It only takes a tiny amount of ligand to half-saturate the protein → **TIGHT binding** (high affinity)
- **Large Kd (e.g., 1 mM)** → It takes LOTS of ligand to half-saturate the protein → **WEAK binding** (low affinity)

**A smaller Kd means tighter binding.** This is the single most important fact to remember.

## 📊 Let's See It: Comparing Different Kd Values

Below, we plot **fraction bound** (θ, from 0 = nothing bound to 1 = fully saturated) vs. **ligand concentration** for three different proteins with different Kd values.

The equation for the fraction bound is:

$$ \theta = \frac{[L]}{K_d + [L]} $$

Run the cell and look at the plot carefully.

In [ ]:
# Create a range of ligand concentrations (from 0.001 to 1000 nM)
# We use a 'logspace' to see behavior across many orders of magnitude
L = np.logspace(-3, 3, 200)  # 200 points from 0.001 to 1000 nM

# Define three different Kd values (in nM)
Kd_tight  = 0.1    # tight binder
Kd_medium = 10     # medium binder
Kd_weak   = 500    # weak binder

# Calculate fraction bound for each
theta_tight  = L / (Kd_tight + L)
theta_medium = L / (Kd_medium + L)
theta_weak   = L / (Kd_weak + L)

# Make the plot
plt.figure(figsize=(9, 5))
plt.semilogx(L, theta_tight,  label=f'Tight binder (Kd = {Kd_tight} nM)',  linewidth=2)
plt.semilogx(L, theta_medium, label=f'Medium binder (Kd = {Kd_medium} nM)', linewidth=2)
plt.semilogx(L, theta_weak,   label=f'Weak binder (Kd = {Kd_weak} nM)',    linewidth=2)

# Add horizontal line at half-saturation to show where Kd is read off
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Half saturation (θ = 0.5)')

plt.xlabel('[Ligand] (nM) — log scale', fontsize=12)
plt.ylabel('Fraction bound (θ)', fontsize=12)
plt.title('Binding curves for three proteins with different Kd values', fontsize=13)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

### 🤔 Think about it:
1. Where does each curve cross the dashed line (θ = 0.5)? That value on the x-axis **is the Kd**.
2. Which protein needs the LEAST ligand to be half-saturated? Which needs the MOST?
3. Which protein has the **highest affinity** for its ligand?

**Answer:** The tight binder (Kd = 0.1 nM) has the highest affinity — a smaller Kd = tighter binding.

## 🎮 Your Turn: Play with Kd

Change the value of `my_Kd` below to any number you want (try 0.01, 1, 100, or 10000) and re-run the cell. Watch how the curve shifts left or right.

In [ ]:
# 👇 CHANGE THIS NUMBER AND RE-RUN THE CELL
my_Kd = 5.0   # in nM — try different values!

L = np.logspace(-3, 4, 200)
theta = L / (my_Kd + L)

plt.figure(figsize=(8, 5))
plt.semilogx(L, theta, 'b-', linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6)
plt.axvline(my_Kd, color='red', linestyle='--', alpha=0.6, label=f'Kd = {my_Kd} nM')
plt.xlabel('[Ligand] (nM) — log scale', fontsize=12)
plt.ylabel('Fraction bound (θ)', fontsize=12)
plt.title(f'Binding curve for Kd = {my_Kd} nM', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"At [L] = Kd = {my_Kd} nM, fraction bound = {my_Kd/(my_Kd + my_Kd):.2f} (should be 0.5!)")

---
# Part 2: Kd and Ka — Two Sides of the Same Coin

The **association constant Ka** describes the *forward* reaction (binding):

$$ K_a = \frac{[PL]}{[P][L]} $$

Notice this is just the **inverse** of Kd:

$$ K_a = \frac{1}{K_d} $$

### Key differences:
| Property | Kd (dissociation) | Ka (association) |
|---|---|---|
| Describes | Falling apart | Coming together |
| Units | Concentration (M, nM, μM…) | 1/concentration (M⁻¹, nM⁻¹…) |
| Tight binding means… | **Small** Kd | **Large** Ka |
| Most common in biochem | ✅ Usually reported | Less common |

### Quick example:
If Kd = 10 nM, then Ka = 1/(10 nM) = 0.1 nM⁻¹ = 10⁸ M⁻¹.

Let's verify this with code:

In [ ]:
# Try a few Kd values and calculate the corresponding Ka
Kd_values_M = [1e-9, 1e-6, 1e-3]   # Kd in molar: 1 nM, 1 μM, 1 mM
labels      = ['1 nM (tight)', '1 μM (medium)', '1 mM (weak)']

print(f"{'Kd':<15}{'Ka (M⁻¹)':<20}{'Binding strength'}")
print("-" * 55)
for Kd, label in zip(Kd_values_M, labels):
    Ka = 1 / Kd
    print(f"{label:<15}{Ka:<20.2e}{'← smaller Kd = larger Ka = tighter'}")

### 📝 Practice Questions (Part 1 & 2)

Try answering these in your head (or on paper) before looking at the answers below.

1. **Protein A** has Kd = 5 nM for ligand X. **Protein B** has Kd = 500 nM for ligand X. Which protein binds ligand X more tightly?
2. If a drug has Kd = 2 nM for its receptor target, roughly what drug concentration do you need in the body to have half the receptors occupied?
3. Ka for a hormone-receptor pair is 10⁹ M⁻¹. What is the Kd? Is this tight or weak binding?

<details>
<summary><b>Click here for answers</b></summary>

1. **Protein A** — smaller Kd means tighter binding.
2. About **2 nM** — half-occupancy occurs when [L] = Kd.
3. Kd = 1/Ka = 1/(10⁹ M⁻¹) = **10⁻⁹ M = 1 nM** — this is **tight binding** (nanomolar affinity).
</details>

---
# Part 3: Reading Kd from Real Binding Data

In the lab, you don't get a smooth curve — you get **data points** from experiments (e.g., fluorescence, ITC, surface plasmon resonance).

Below is a simulated dataset for a protein binding a ligand. The data has some noise, like real experiments. **Look at the plot and estimate the Kd** before we calculate it exactly.

In [ ]:
# Simulated experimental data — pretend this is data you collected in the lab
ligand_concentrations = np.array([0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000])  # in nM
fraction_bound_data   = np.array([0.02, 0.06, 0.18, 0.35, 0.68, 0.85, 0.94, 0.97, 0.99])

# Plot the data
plt.figure(figsize=(8, 5))
plt.semilogx(ligand_concentrations, fraction_bound_data, 'ko', markersize=9, label='Experimental data')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='θ = 0.5')
plt.xlabel('[Ligand] (nM)', fontsize=12)
plt.ylabel('Fraction bound (θ)', fontsize=12)
plt.title('Simulated binding data — estimate the Kd!', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("🔍 Look at the plot: at what [Ligand] does the curve appear to cross θ = 0.5?")
print("   That's your Kd estimate!")

### 🎯 Data Interpretation Questions

1. Based on the plot above, roughly what is the Kd?
2. Would you call this tight, medium, or weak binding?
3. At [L] = 1000 nM, is the protein essentially saturated?
4. What would happen to the curve if the Kd were 10× smaller? 10× larger?

<details>
<summary><b>Click for answers</b></summary>

1. Kd is approximately **5–7 nM** (where the curve crosses 0.5).
2. This is **medium-tight binding** — nanomolar range is typical for biological interactions.
3. Yes — at [L] > 100× Kd, the protein is essentially fully bound (>99%).
4. Smaller Kd → curve shifts **left**; larger Kd → curve shifts **right**.
</details>

---
# Part 4: Fitting Your Own Binding Curve

Eyeballing Kd is a good first step, but scientists use **curve fitting** to get a precise value. This means we ask the computer to find the Kd value that makes the theoretical curve fit our data points as closely as possible.

## The Model

We fit the data to the standard binding equation:

$$ \theta = \frac{[L]}{K_d + [L]} $$

The computer will find the best Kd for us. Here's how:

In [ ]:
# Step 1: Define the binding equation as a Python function
# This says: given a ligand concentration L and a Kd, calculate fraction bound
def binding_equation(L, Kd):
    return L / (Kd + L)

# Step 2: Use curve_fit to find the best Kd
# It compares our data to the equation and finds the best-fit Kd
best_fit_params, _ = curve_fit(binding_equation, ligand_concentrations, fraction_bound_data, p0=[10])

# Extract the fitted Kd value
fitted_Kd = best_fit_params[0]
print(f"🎉 Best-fit Kd = {fitted_Kd:.2f} nM")

In [ ]:
# Step 3: Plot the data with our fitted curve on top
L_smooth = np.logspace(-2, 4, 200)  # smooth line for plotting
theta_fit = binding_equation(L_smooth, fitted_Kd)

plt.figure(figsize=(9, 5))
plt.semilogx(ligand_concentrations, fraction_bound_data, 'ko', markersize=10, label='Experimental data')
plt.semilogx(L_smooth, theta_fit, 'b-', linewidth=2, label=f'Best fit: Kd = {fitted_Kd:.2f} nM')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.axvline(fitted_Kd, color='red', linestyle='--', alpha=0.5, label=f'Kd = {fitted_Kd:.2f} nM')
plt.xlabel('[Ligand] (nM)', fontsize=12)
plt.ylabel('Fraction bound (θ)', fontsize=12)
plt.title('Binding curve with best-fit Kd', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 🧪 Your Turn: Fit Your Own Data!

Now it's your turn. Enter your own binding data below (or use the data your instructor provides) and let the computer fit it.

**Instructions:**
1. Replace the numbers in `my_ligand_concentrations` with YOUR ligand concentrations (in nM).
2. Replace the numbers in `my_fraction_bound` with YOUR measured fraction bound values.
3. **Both lists must have the same number of values!**
4. Run the cell and see your fitted Kd.

In [ ]:
# 👇 REPLACE THESE NUMBERS WITH YOUR OWN DATA 👇

my_ligand_concentrations = np.array([0.5, 1, 2, 5, 10, 20, 50, 100, 200])   # in nM
my_fraction_bound        = np.array([0.05, 0.11, 0.20, 0.38, 0.55, 0.72, 0.86, 0.93, 0.96])

# ---- Everything below just runs the fit and makes the plot ----

# Fit the data
best_fit, _ = curve_fit(binding_equation, my_ligand_concentrations, my_fraction_bound, p0=[10])
my_Kd_fit = best_fit[0]
my_Ka_fit = 1 / (my_Kd_fit * 1e-9)   # convert nM to M for Ka

# Plot
L_smooth = np.logspace(np.log10(my_ligand_concentrations.min()/5),
                       np.log10(my_ligand_concentrations.max()*5), 200)
theta_smooth = binding_equation(L_smooth, my_Kd_fit)

plt.figure(figsize=(9, 5))
plt.semilogx(my_ligand_concentrations, my_fraction_bound, 'ro', markersize=10, label='Your data')
plt.semilogx(L_smooth, theta_smooth, 'g-', linewidth=2, label=f'Best fit: Kd = {my_Kd_fit:.2f} nM')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.axvline(my_Kd_fit, color='red', linestyle='--', alpha=0.5)
plt.xlabel('[Ligand] (nM)', fontsize=12)
plt.ylabel('Fraction bound (θ)', fontsize=12)
plt.title('Your binding curve fit', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"📊 RESULTS FROM YOUR DATA:")
print(f"   Kd = {my_Kd_fit:.3f} nM")
print(f"   Ka = {my_Ka_fit:.2e} M⁻¹")
print(f"   Binding strength: " +
      ('very tight (sub-nM)' if my_Kd_fit < 1 else
       'tight (nM)' if my_Kd_fit < 100 else
       'moderate (sub-μM)' if my_Kd_fit < 1000 else
       'weak (μM or higher)'))

---
# 🎓 Wrap-Up: What You've Learned

You now understand:

✅ **Kd is the ligand concentration at which half the protein is bound** — it has units of concentration (usually nM or μM).

✅ **Smaller Kd = tighter binding = higher affinity.**

✅ **Ka = 1/Kd** — they describe the same equilibrium but from opposite directions.

✅ **You can read Kd off a binding curve** by finding where fraction bound = 0.5.

✅ **You can fit real data to the binding equation** using Python and extract a precise Kd.

### Typical Kd values in biology:
| System | Typical Kd | Binding |
|---|---|---|
| Antibody–antigen (mature) | 0.01 – 10 nM | Very tight |
| Hormone–receptor | 0.1 – 10 nM | Tight |
| Enzyme–substrate (Km ≈ Kd) | 1 μM – 1 mM | Moderate to weak |
| Transient signaling interactions | 1 – 100 μM | Weak |

### 🌟 Challenge yourself:
- Change the noise or number of data points in Part 4. Does the fit still work?
- What happens if all your data points are at ligand concentrations way BELOW the Kd? Or way ABOVE? (Hint: you need points *bracketing* the Kd to get a good fit!)
- Look up the Kd for a real drug (e.g., a kinase inhibitor). Was it tight or weak binding?

**Great work! You now understand one of the most fundamental quantitative concepts in biochemistry.** 🧬